# Crop Health Disease Detection - Inference

This workflow processes crop images through a CNN-based disease detection pipeline. It fetches crop images, preprocesses them for machine learning, trains a disease classification model, runs inference, and generates treatment recommendations. This particular version is a simplified version for use in the ACCESS Pegasus tutorial. The full workflow can be found in [GitHub](https://github.com/pegasus-isi/crophealth-workflow). The changes include:

1. The **training** and **inference** steps are broken up into separate notebooks
2. The **inference** workflow takes a set of individual images as input - which similate a use case of using a trained model over and over again, and highlights using the Pegasus API to loop over a set of inputs.

**Training Workflow Jobs:**
1. `preprocess_images` - Resize, normalize, augment images and split into train/validation sets
2. `train_classifier` - Train a CNN disease classifier using PyTorch (with sklearn fallback)

**Inference Workflow Jobs:**
1. `classify_disease` - Run inference on images and generate disease predictions with confidence scores

## 1. Create the Crop Health Workflow

The following cell defines the `CropHealthWorkflow` class, which builds the Pegasus workflow DAG with site, transformation, and replica catalogs. It then generates and writes the workflow file.

In [ ]:
import os
import sys
import logging
import argparse
from pathlib import Path

from Pegasus.api import *

logging.basicConfig(level=logging.DEBUG)


class CropHealthWorkflow:
    """Generate Pegasus workflow for crop disease detection."""

    wf = None
    sc = None
    tc = None
    rc = None
    props = None

    dagfile = None
    wf_dir = None
    shared_scratch_dir = None
    local_storage_dir = None
    wf_name = "crophealth"

    def __init__(self, dagfile="workflow.yml"):
        """Initialize workflow."""
        self.dagfile = dagfile
        self.wf_dir = str(Path(".").resolve())
        self.shared_scratch_dir = os.path.join(self.wf_dir, "scratch")
        self.local_storage_dir = os.path.join(self.wf_dir, "output")

    def write(self):
        """Write all catalogs and workflow to files."""
        if self.sc is not None:
            self.sc.write()
        self.props.write()
        self.rc.write()
        self.tc.write()
        try:
            self.wf.write(file=self.dagfile)
        except PegasusClientError as e:
            print(e)

    def plan_submit(self):
        """Plan and submit the workflow."""
        try:
            self.wf.plan(submit=True)
        except PegasusClientError as e:
            print(e)

    def status(self):
        """Get workflow status."""
        try:
            self.wf.status(long=True)
        except PegasusClientError as e:
            print(e)

    def wait(self):
        """Wait for workflow completion."""
        try:
            self.wf.wait()
        except PegasusClientError as e:
            print(e)

    def statistics(self):
        """Get workflow statistics."""
        try:
            self.wf.statistics()
        except PegasusClientError as e:
            print(e)

    def create_pegasus_properties(self):
        """Create Pegasus properties configuration."""
        self.props = Properties()
        self.props["pegasus.transfer.threads"] = "16"

    def create_sites_catalog(self, exec_site_name="condorpool"):
        """Create site catalog."""
        self.sc = SiteCatalog()

        local = Site("local").add_directories(
            Directory(
                Directory.SHARED_SCRATCH, self.shared_scratch_dir
            ).add_file_servers(
                FileServer("file://" + self.shared_scratch_dir, Operation.ALL)
            ),
            Directory(
                Directory.LOCAL_STORAGE, self.local_storage_dir
            ).add_file_servers(
                FileServer("file://" + self.local_storage_dir, Operation.ALL)
            ),
        )

        exec_site = (
            Site(exec_site_name)
            .add_condor_profile(universe="vanilla")
            .add_pegasus_profile(style="condor")
        )

        self.sc.add_sites(local, exec_site)

    def create_replica_catalog(self):
        """Create replica catalog for input files."""
        self.rc = ReplicaCatalog()

    def create_transformation_catalog(self, exec_site_name="condorpool"):
        """Create transformation catalog with executables and containers."""
        self.tc = TransformationCatalog()

        crophealth_container = Container(
            "crophealth_container",
            container_type=Container.SINGULARITY,
            image=f"https://download.pegasus.isi.edu/tutorial/crophealth/crophealth-container.sif",
            image_site="www",
        )

        classify_disease = Transformation(
            "classify_disease",
            site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/classify_disease.py"),
            is_stageable=True,
            container=crophealth_container,
        ).add_pegasus_profile(gpus="1")\
         .add_pegasus_profile(memory="28 GB")

        generate_report = Transformation(
            "generate_report",
            site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/generate_report.py"),
            is_stageable=True,
            container=crophealth_container,
        ).add_pegasus_profile(memory="28 GB")

        self.tc.add_containers(crophealth_container)
        self.tc.add_transformations(
            classify_disease
        )

    def create_workflow(self):
        """Create the complete workflow."""
        self.wf = Workflow(self.wf_name)

        # common inputs
        model_checkpoint = File("disease_classifier.pt")
        training_info = File("training_info.json")
        
        # input locations - these would be generated by the previous training job, but we provide
        # standalone inputs here to that the inference workflow is independent from the training
        # workflow
        self.rc.add_replica("remote", model_checkpoint, f"https://download.pegasus.isi.edu/tutorial/crophealth/{model_checkpoint}")
        self.rc.add_replica("remote", training_info, f"https://download.pegasus.isi.edu/tutorial/crophealth/{training_info}")

        # sample input images from https://download.pegasus.isi.edu/tutorial/crophealth/inference/
        for image_num in range(10):
            image_id = f"{image_num:02d}"
            
            # inputs
            image_file = File(f"{image_id}.jpg")
            self.rc.add_replica("remote", image_file, f"https://download.pegasus.isi.edu/tutorial/crophealth/inference/{image_file}")
            
            # outputs
            predictions_file = File(f"{image_id}_predictions.json")

            # Job: Classify diseases
            classify_job = Job("classify_disease", _id=f"classify_{image_id}", node_label=f"classify_{image_id}")
            classify_job.add_args(
                "--input", image_file,
                "--output", predictions_file
            )
            classify_job.add_inputs(model_checkpoint, training_info, image_file)
            classify_job.add_outputs(predictions_file, stage_out=True, register_replica=False)
            
            # add job
            self.wf.add_jobs(classify_job)



# --- Build and generate the workflow ---
dagfile = 'workflow.yml'

workflow = CropHealthWorkflow(dagfile=dagfile)

print("Creating execution sites...")
workflow.create_sites_catalog("condorpool")

print("Creating workflow properties...")
workflow.create_pegasus_properties()

print("Creating transformation catalog...")
workflow.create_transformation_catalog("condorpool")

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating crop health workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nCrop Health Workflow has been generated!")

## 2. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. The graph shows the linear pipeline: fetch images, preprocess, train classifier, classify diseases, and generate report.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image
Image(filename='workflow.png')

## 3. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the **condorpool** site (the selected ACCESS resource).

In [ ]:
workflow.plan_submit()

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed states.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 4. Statistics

After the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 5. Examining the Results

The workflow produces the following outputs in the `output/` directory:

| File | Description |
|------|-------------|
| `NN_predictions.json` | Disease predictions with confidence scores and treatment recommendations |

In [ ]:
!ls -ltR output/

Let's look at the results for one of the images. Remember, this is what the input looks like:

<img src="https://download.pegasus.isi.edu/tutorial/crophealth/inference/00.jpg"/>

And this is what our trained model came up with:

In [ ]:
!cat output/00_predictions.json